# Exploring Pretrained Base Models

> **Optional extension:** You do not need this notebook to train or run DustyLM. If you want to build the complete 8M model from scratch, start with the [Train from Scratch notebook](02_train_from_scratch.ipynb).

DustyLM's 8M model is intentionally small so you can inspect and train every part of it. When you need broader language patterns and more capacity, you can keep the same local training code while starting from a pretrained model.

This notebook uses Hugging Face's [SmolLM2](https://huggingface.co/collections/HuggingFaceTB/smollm2-6723884218bc2d4ccd3223ec) 135M and 360M base models. You will inspect their profiles, convert the official weights into DustyLM's local checkpoint format, generate a continuation, and see how optional Supervised Fine-Tuning (SFT) can teach the same Dusty persona to a larger model.

SmolLM2 is stronger than the 8M tutorial model, but it is still a compact model with real capability limits. Treat its output as generated text, not as a reliable source of facts.


## Setup

**Google Colab (recommended):**

1. Open **Runtime > Change runtime type**.
2. Select **T4 GPU**, then click **Save**.
3. Run the Setup cell below, then continue through the notebook.

**RunPod:** Start a GPU pod, then run the Setup cell below.

**Local laptop:**

1. If needed, [install uv](https://docs.astral.sh/uv/getting-started/installation/), then run `uv sync` from the repository root.
2. Select the project's `.venv` as your notebook kernel.
3. Run the Setup cell below.

In [ ]:
import os
import sys
from pathlib import Path

is_cloud = "google.colab" in sys.modules or "RUNPOD_POD_ID" in os.environ
if is_cloud:
    !pip install -q uv
    if not Path("Makefile").exists():
        !git clone --depth 1 https://github.com/khordoo/dusty-lm.git
        os.chdir("dusty-lm")
    !pip install -q -e .
elif not Path("Makefile").exists():
    # Walk up so make commands find the Makefile from any CWD
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "Makefile").exists():
            os.chdir(str(parent))
            break
    else:
        raise FileNotFoundError("Open this notebook from inside the dusty-lm repository.")

print(f"✅ Setup complete. Repo root: {Path.cwd()}")




## 1. Understanding Profiles in DustyLM

DustyLM uses **profiles** to connect a model architecture with its tokenizer, checkpoints, generation defaults, and optional training settings. The profiles live in [`dustylm/config.py`](../dustylm/config.py).

There are two main categories of profiles:

* **From-scratch profiles**, such as `dusty8m` and `sft_dusty8m`, define the small architecture you train yourself. You can experiment with its dimensions and tokenizer.
* **Pretrained profiles**, such as `smollm2_135m`, reproduce an external model's architecture and connect it to downloadable weights.

> ⚠️ **Pretrained profile rule:** Architecture settings must match the source checkpoint, including layer dimensions, attention heads, RoPE settings, and normalization epsilon. Changing them means you are no longer running the same pretrained model.

You can still change generation controls such as temperature and top-p, or training settings such as batch size and learning rate.

### 1.1 Inspect the Profiles

Compare the tutorial model with the two pretrained base profiles and the optional SmolLM2 SFT profile.

In [ ]:
from dustylm.config import get_profile

for name in ["dusty8m", "smollm2_135m", "smollm2_360m", "sft_smollm2_135m"]:
    profile = get_profile(name)
    model = profile.model
    print("=" * 80)
    print("profile:", profile.name)
    if profile.description:
        print("description:", profile.description)
    print("family:", model.family)
    print("layers:", model.num_layers)
    print("hidden dim:", model.embed_dim)
    print("heads:", model.num_heads, "query /", model.num_kv_heads, "kv")
    print("vocab:", model.vocab_size)
    print("RoPE base:", model.rope_base)
    print("RMS norm epsilon:", model.rms_eps)
    print("tokenizer:", model.tokenizer.path_or_name)
    if profile.training:
        print("training dataset:", profile.training.dataset_path)
    if profile.generation:
        print("checkpoint:", profile.generation.checkpoint_path)
    if profile.hf_artifacts:
        print("hf repo:", profile.hf_artifacts.repo_id)
        print("hf weights cache:", profile.hf_artifacts.local_weights_path)

---

## 2. Scaling Up: Importing External Weights

The `dusty8m` profile learns language patterns from the tutorial dataset. SmolLM2 starts from weights trained on a much larger and more diverse corpus.

According to the official model cards, [SmolLM2 135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) was pretrained on 2 trillion tokens using 64 H100 GPUs, while [SmolLM2 360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M) used 4 trillion tokens and 128 H100 GPUs. DustyLM maps those published tensors into its readable local SmolLM2 implementation.

> ⚠️ **Keep the pretrained tokenizer:** The embedding rows correspond to fixed token IDs. The download command therefore saves the matching tokenizer to `artifacts/tokenizers/smollm2_tokenizer.json`; do not replace it with the Dusty tokenizer.

### 2.1 Download and Convert a Base Checkpoint

`dustylm.artifacts download --convert` does two things:

1. Downloads the pinned Hugging Face weights and matching tokenizer.
2. Converts the weights into a DustyLM `.pt` checkpoint under `artifacts/checkpoints/`.

Start with the 135M model. You can change `PROFILE` to `smollm2_360m` later if you want to try the larger base model.

In [ ]:
PROFILE = "smollm2_135m"  # Change to "smollm2_360m" to try the larger base model.

!uv run python -m dustylm.artifacts download --profile {PROFILE} --convert

## 3. Confirm Artifact Paths

After conversion, generation reads the `.pt` checkpoint from `artifacts/checkpoints/`.

In [ ]:
from pathlib import Path
from dustylm.config import get_profile

profile = get_profile(PROFILE)
paths = [
    profile.hf_artifacts.local_weights_path,
    profile.hf_artifacts.local_tokenizer_path,
    profile.generation.checkpoint_path,
]

missing = []
for path in map(Path, paths):
    exists = path.exists()
    print("✅" if exists else "❌", path)
    if not exists:
        missing.append(path)

if missing:
    raise FileNotFoundError(f"Missing artifacts: {missing}")

## 4. Generate From the Pretrained Base Model

Notice that this model doesn't have the "Dusty" personality yet. It is simply the raw SmolLM2 base model running through DustyLM's local generation stack.

In [ ]:
!uv run python -m dustylm.generate \
  --profile {PROFILE} \
  --prompt "A robot vacuum found a crumb under the couch and" \
  --max-new-tokens 128 \
  --temperature 0.8 \
  --top-p 0.9


## 5. Optional: Fine-Tune the Dusty Persona

The base model can continue text, but it has not learned to respond as Dusty. SFT teaches that response style using the same raw Dusty conversations as the 8M model.

This section is optional. The repository currently includes an SFT profile for 135M only. The 360M profile supports base-model conversion and generation, but fine-tuning it requires adding a separate SFT profile. Full fine-tuning updates all 135M parameters, takes much longer than the 8M tutorial, and is best run on a GPU. The default notebook path ends after base-model generation above.

### 5.1 Prepare SmolLM2 SFT Data

The raw conversations can be shared, but token IDs cannot. `make data-sft` remains the default Dusty 8M command. For this model, `make data-sft-smollm2` uses the pretrained SmolLM2 tokenizer and writes to a separate directory:

`artifacts/datasets/dusty_sft_smollm2_tokenized`

Run the next cell only if you want to fine-tune SmolLM2. It downloads the raw Dusty conversations when they are missing, then prepares them with the correct tokenizer.


In [ ]:
from pathlib import Path

from data_pipeline.download_datasets import download_dusty_sft
from dustylm.config import get_profile

sft_profile = get_profile("sft_smollm2_135m")
raw_sft_path = Path(sft_profile.training.raw_sft_path)
if not raw_sft_path.exists():
    download_dusty_sft(
        repo_id="mkhordoo/dusty-chat",
        filename="dusty_sft.jsonl",
        output_path=raw_sft_path,
    )

!make data-sft-smollm2


### 5.2 Run the Optional Fine-Tune

The settings below were tested on a free Colab T4. Eight epochs with batch size 128 completed in about 20 minutes and produced the most reliable persona responses. For a faster pipeline check, use three epochs, which takes about 7.5 minutes. Runtime varies with the assigned GPU. If batch size 128 does not fit on your hardware, reduce it to 64.

Saving every 500 steps creates four intermediate checkpoints during the eight-epoch run. Each checkpoint is roughly 513 MB, so the intermediate files use about 2 GB in addition to the final checkpoint. Set `--checkpoint-every-steps 0` if you only want the final model.

Optionally run the TensorBoard cell below before training. It opens a non-blocking dashboard that updates as the loss is recorded.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir artifacts/tensorboard

Uncomment and run the following cell to train the model.

In [ ]:
# !uv run python -m dustylm.train --profile sft_smollm2_135m --epochs 8 --batch-size 128 --checkpoint-every-steps 500


### 5.3 Chat With Your Fine-Tuned Model

Run the next cell after training. It opens the notebook-native chat when the fine-tuned checkpoint exists; otherwise, it tells you which training file is missing. You can rerun this cell later without retraining or replacing your checkpoint.

In [ ]:
from pathlib import Path

from dustylm.config import get_profile

chat_profile = get_profile("sft_smollm2_135m")
checkpoint_path = Path(chat_profile.generation.checkpoint_path)

if checkpoint_path.exists():
    from dustylm.inference import Inference

    print("Loading your fine-tuned model...")
    chat_engine = Inference(profile_name="sft_smollm2_135m")
    messages = []
    stop_words = {"q", "quit", "exit", "bye"}

    print("\nChat with your fine-tuned model.")
    print(f"Maximum conversation history: {chat_engine.spec.max_chat_turns} user turns.")
    print("Type q, quit, exit, or bye to stop.\n")
    while True:
        user_text = input("You> ").strip()
        if user_text.lower() in stop_words:
            break
        if not user_text:
            continue

        messages.append({"role": "user", "content": user_text})
        response = chat_engine.chat_completion(
            messages, temperature=0.7, max_tokens=64, top_p=1.0
        )
        assistant_text = response["choices"][0]["message"]["content"]
        print(f"Dusty> {assistant_text}\n")
        messages.append({"role": "assistant", "content": assistant_text})
else:
    print("Fine-tuned checkpoint not found:")
    print(f"  {checkpoint_path}")
    print("Run the training cell above, then rerun this chat cell.")


### 5.4 Use Your Own Persona Data

First create and review the training conversations for your character by following **Create a New Persona** in the [Advanced Tools notebook](03_advanced_tools.ipynb). Save the accepted conversations as a JSONL file.

Then follow these steps:

1. **Update the profile.** Open [`dustylm/config.py`](../dustylm/config.py), locate `sft_smollm2_135m`, and update these lines:

    ```python
    raw_sft_path=REPO_ROOT / "artifacts" / "datasets" / "my_persona.jsonl",
    sft_user_field="user",
    sft_assistant_field="dusty",
    ```

    Keep `user` and `dusty` if you used the Advanced Tools generator. Change them only if your JSONL uses different field names.

2. **Tokenize the new dataset.** This converts the conversations with the SmolLM2 tokenizer:

    ```bash
    make data-sft-smollm2
    ```

3. **Train the model.** Return to section 5.2 and run the training cell.

## 6. Practical Guidance

- Use `dusty8m` when you want to learn the full stack from scratch.
- Use `smollm2_135m` when you want a stronger pretrained base with a manageable local footprint.
- Use `smollm2_360m` when you have more memory and disk space.
- Keep SmolLM2 artifacts under `artifacts/hf/`, `artifacts/tokenizers/`, and `artifacts/checkpoints/`.
- Keep Dusty and SmolLM2 tokenized datasets separate because their token IDs are not interchangeable.
- Use profiles to keep model, tokenizer, checkpoint, and training settings together.